# 00 · Orientation — Thinking in Systems

**Part 0 · Foundations** · *Book mapping: DDIA ch. 1*

Welcome. This course teaches you to design **data-intensive systems** — systems whose hard
problems come from data (its volume, its rate of change, its complexity) rather than from CPU
cycles — and **machine learning systems**, which are data-intensive systems with a model in the
loop.

**Learning objectives.** After this notebook you can:

1. Name the three concerns every system design must answer to — reliability, scalability,
   maintainability — and define each precisely.
2. Compute availability for serial and redundant compositions, and explain why redundancy stops
   helping when failures are correlated.
3. Explain why *percentiles* (not averages) describe latency, and quantify **tail latency
   amplification**: how fan-out turns rare slowness into common slowness.
4. Apply the design framework used in every later notebook.

**How this course works — three rules:**

- **Everything is built.** Concepts are implemented as small, working Python systems. The point
  is not production code; it is that a mechanism you have built cannot stay a buzzword.
- **Failure first.** Where possible we reproduce the *problem* (a stale read, a lost update, a
  leaking feature) before building the fix. You remember what you have watched go wrong.
- **Self-verifying.** Claims are backed by inline `assert`s. If a claim stops being true when
  you tinker — and you should tinker — the notebook tells you loudly.

Prerequisites: comfortable Python, basic probability. Run cells top to bottom.

## 0. Environment check

Fail loudly *now* rather than confusingly later.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sysdes
from sysdes import viz

# The course targets the "sysdes" conda environment (see environment.yml).
assert sys.version_info >= (3, 12), f"need Python >= 3.12, got {sys.version}"

# One shared, seeded random generator per notebook: every run of this notebook
# produces identical numbers, which is what makes the inline asserts possible.
rng = np.random.default_rng(seed=0)

viz.use_style()  # consistent plot style across the whole course
print(f"python {sys.version.split()[0]} · numpy {np.__version__} · sysdes {sysdes.__version__}")

## 1. What is a data-intensive application?

Most services are not built from scratch; they are *composed* from a small set of standard
building blocks:

- **Database** — the *system of record*: stores data so it can be found and updated later.
- **Cache** — remembers results of expensive operations so repeats are fast (and can be *stale*).
- **Index** — a structure for finding data by content (search, secondary lookups).
- **Queue / log** — buffers messages so work can happen later or elsewhere (*asynchronously*).
- **Stream / batch processor** — crunches data continuously or periodically to produce
  *derived data* (aggregates, features, recommendations).

System design is deciding **which blocks to use, how to wire them, and what guarantees the
wiring gives you** — under your specific workload, budget, and failure assumptions. A typical
composition:

In [ ]:
fig, ax = viz.system_diagram(
    nodes=[
        {"id": "client", "x": 0.0, "y": 1.0, "label": "clients", "kind": "client"},
        {"id": "api", "x": 2.2, "y": 1.0, "label": "app servers\n(stateless)"},
        {"id": "cache", "x": 4.6, "y": 2.0, "label": "cache", "kind": "cache"},
        {"id": "db", "x": 4.6, "y": 0.9, "label": "primary DB", "kind": "store"},
        {"id": "queue", "x": 4.6, "y": -0.4, "label": "queue", "kind": "queue"},
        {"id": "worker", "x": 6.9, "y": -0.4, "label": "workers"},
        {"id": "index", "x": 9.0, "y": 0.3, "label": "search index", "kind": "store"},
        {"id": "dwh", "x": 9.0, "y": -1.1, "label": "analytics\nstore", "kind": "store"},
    ],
    edges=[
        {"src": "client", "dst": "api", "label": "requests"},
        {"src": "api", "dst": "cache", "label": "read-through", "bidir": True},
        {"src": "api", "dst": "db", "label": "reads/writes", "bidir": True},
        {"src": "api", "dst": "queue", "label": "async tasks"},
        {"src": "queue", "dst": "worker", "label": ""},
        {"src": "worker", "dst": "index", "label": "update"},
        {"src": "worker", "dst": "dwh", "label": "events"},
    ],
    title="A typical data-intensive application: standard blocks, custom wiring",
)
plt.show()

Every arrow in that picture is a place where things can be slow, lost, duplicated, or stale.
The rest of the course is about exactly those arrows. Three concerns organize all of it.

## 2. Reliability

> **Reliability**: the system continues to work *correctly* (right results, target performance)
> even when things go wrong.

The things that go wrong are **faults** — one component deviating from its spec. A **failure**
is when the *system as a whole* stops providing its service. The whole game is building
**fault-tolerant** systems: arranging components so faults do not become failures.

Three fault families, in increasing order of nastiness:

| Fault family | Character | Main defense |
|---|---|---|
| **Hardware** (disk dies, RAM flips, power cut) | Random, mostly *independent* | Redundancy: more copies |
| **Software** (bug, runaway process, cascading overload) | Systematic, *correlated* — hits every replica running the same code | Testing, isolation, rollbacks, backpressure |
| **Human** (bad config push, fat-fingered delete) | The leading cause of outages | Good interfaces, sandboxes, gradual rollout, fast recovery, telemetry |

### 2.1 The arithmetic of redundancy

**Availability** is the fraction of time a system is up. It is quoted in "nines":

In [ ]:
HOURS_PER_YEAR = 365.25 * 24

nines = pd.DataFrame({"availability": [0.9, 0.99, 0.999, 0.9999, 0.99999]})
nines["downtime per year"] = [
    # Convert the unavailable fraction into a human-readable duration.
    f"{(1 - a) * HOURS_PER_YEAR:.1f} h" if (1 - a) * HOURS_PER_YEAR >= 1
    else f"{(1 - a) * HOURS_PER_YEAR * 60:.1f} min"
    for a in nines["availability"]
]
nines.style.format({"availability": "{:.3%}"}).hide(axis="index")

Two composition rules cover most back-of-envelope reliability math. With independent component
availabilities $a_i$:

- **Serial** (every component must be up — a chain of dependencies):
  $A_\text{serial} = \prod_i a_i$ — availability *degrades* as chains grow.
- **Parallel** (at least one replica must be up):
  $A_\text{parallel} = 1 - \prod_i (1 - a_i)$ — unavailability *multiplies away*.

In [ ]:
def serial(*avail: float) -> float:
    """Availability of a dependency chain: all components must be up."""
    return float(np.prod(avail))

def parallel(*avail: float) -> float:
    """Availability of a redundant group: at least one replica must be up."""
    return 1.0 - float(np.prod([1 - a for a in avail]))

# A chain of three well-run (99.9%) services is noticeably worse than any of them:
chain = serial(0.999, 0.999, 0.999)
# ...while just two mediocre (99%) replicas beat a single excellent one:
duo = parallel(0.99, 0.99)

print(f"3 x 99.9% in series   -> {chain:.4%}  ({(1-chain)*HOURS_PER_YEAR:.1f} h down/year)")
print(f"2 x 99.0% in parallel -> {duo:.4%}  ({(1-duo)*HOURS_PER_YEAR:.1f} h down/year)")

assert chain < 0.999, "chains are weaker than their weakest link count suggests"
assert duo > 0.999, "cheap redundancy beats expensive single components"

### 2.2 Failure first: when redundancy stops working

The parallel formula assumes faults are **independent**. Real fleets share things: a top-of-rack
switch, a bad kernel version, one config file. A **correlated** (common-mode) fault takes out
every replica at once — and no amount of redundancy helps.

Let's verify both regimes with a Monte-Carlo simulation: sample each hour of many simulated
years, mark each replica up or down, and measure system availability.

In [ ]:
def simulated_availability(n_replicas: int, replica_avail: float,
                           shared_avail: float = 1.0, n_hours: int = 400_000) -> float:
    """Fraction of hours the system is up.

    Each hour: every replica is independently up with prob `replica_avail`,
    AND a shared dependency (switch/config/code path) is up with prob
    `shared_avail`. The system is up iff the shared dependency is up and at
    least one replica is up.
    """
    replicas_up = rng.random((n_hours, n_replicas)) < replica_avail
    shared_up = rng.random(n_hours) < shared_avail
    return float(np.mean(shared_up & replicas_up.any(axis=1)))

replica_counts = np.arange(1, 7)
independent = [simulated_availability(n, replica_avail=0.99) for n in replica_counts]
correlated  = [simulated_availability(n, replica_avail=0.99, shared_avail=0.999)
               for n in replica_counts]

# The simulation must agree with the closed-form math (within Monte-Carlo noise):
assert abs(independent[1] - parallel(0.99, 0.99)) < 0.001, "simulation vs formula mismatch"
# With a 99.9%-available shared dependency, availability plateaus at ~99.9%
# no matter how many replicas we add:
assert correlated[5] < 0.9995, "correlated faults cap availability at the shared dependency"
assert independent[5] > 0.999999, "independent faults are crushed by redundancy"

fig, ax = plt.subplots()
ax.plot(replica_counts, [1 - a for a in independent], "o-", label="independent faults")
ax.plot(replica_counts, [1 - a for a in correlated], "s-",
        label="+ shared dependency (99.9%)")
ax.set_yscale("log")
ax.set_xlabel("number of replicas")
ax.set_ylabel("unavailability (log scale)")
ax.set_title("Redundancy crushes independent faults — and hits a wall on correlated ones")
ax.legend()
plt.show()

The orange plateau is one of the most important pictures in system design: **your availability
is capped by your least-available shared dependency.** This is why later notebooks obsess over
removing single points of failure (leaderless replication, partitioned brokers) — and why
software and human faults, which are inherently correlated, need entirely different defenses
(isolation, gradual rollouts, fast recovery) rather than more copies.

## 3. Scalability

> **Scalability**: the system's ability to keep performing as **load** grows — described not by
> a yes/no label but by curves: *if load doubles, what happens to performance?*

To talk about it precisely you need two vocabularies:

- **Load parameters** — numbers that describe your workload: requests/second, read/write ratio,
  concurrent users, fan-out (how many downstream items one request touches), working-set size.
  Which parameter matters is system-specific: Twitter's hard problem was not tweet *volume*
  (~12k writes/s at the time) but *fan-out* — each tweet may need delivery to millions of
  follower timelines. We build exactly that in notebook 23.
- **Performance metrics** — for online systems, **response time**: the time a client waits.
  It is a *distribution*, not a number.

### 3.1 Percentiles, not averages

The **p-th percentile** is the value below which p% of observations fall: p50 (median) is the
typical case, p99 is the 1-in-100 worst case. Averages hide the tail; users live in it — and
the slowest requests often belong to your most valuable users (they have the most data).

In [ ]:
# Response times are typically right-skewed: most requests fast, a long tail slow.
# A lognormal distribution is a reasonable stand-in (median ~20 ms here).
latency_ms = rng.lognormal(mean=3.0, sigma=0.55, size=200_000)

stats = {p: float(np.percentile(latency_ms, p)) for p in (50, 95, 99, 99.9)}
mean = float(latency_ms.mean())
print(f"mean {mean:5.1f} ms | " +
      " | ".join(f"p{p:g} {v:6.1f} ms" for p, v in stats.items()))

# The mean sits ABOVE the median (the tail drags it up), yet far UNDER the tail
# itself — it describes nobody's experience:
assert stats[50] < mean < stats[95], "the mean describes neither typical nor bad requests"
assert stats[99.9] > 4 * stats[50], "the tail is multiples of the typical case"

fig, ax = plt.subplots()
ax.hist(latency_ms, bins=np.linspace(0, 150, 120), color="#90caf9")
for label, v, color in [("p50", stats[50], "#2e7d32"), ("mean", mean, "#616161"),
                        ("p99", stats[99], "#c62828")]:
    ax.axvline(v, color=color, linestyle="--")
    ax.text(v, ax.get_ylim()[1] * 0.9, f" {label}", color=color)
ax.set_xlabel("response time (ms)")
ax.set_ylabel("requests")
ax.set_title("Latency is a distribution — summarize it with percentiles")
plt.show()

### 3.2 Failure first: tail latency amplification

Modern requests **fan out**: one page load queries N backends (services, partitions, cache
shards) and must wait for the *slowest* one. That changes everything about the tail:

$$P(\text{request touches the tail}) = 1 - (1 - p)^N$$

With p99-slowness ($p = 0.01$) and a fan-out of $N = 100$, that is $1 - 0.99^{100} \approx 63\%$
of *all* requests. Rare slowness becomes the common case:

In [ ]:
p99_single = stats[99]

def fanout_latency(n_backends: int, n_requests: int = 40_000) -> np.ndarray:
    """Latency of requests that must wait for ALL of n_backends parallel calls."""
    draws = rng.lognormal(mean=3.0, sigma=0.55, size=(n_requests, n_backends))
    return draws.max(axis=1)

fanouts = [1, 2, 5, 10, 30, 100]
touched = {}   # fraction of requests slower than the single-backend p99
medians = {}   # median end-to-end latency
for n in fanouts:
    lat = fanout_latency(n)
    touched[n] = float(np.mean(lat > p99_single))
    medians[n] = float(np.median(lat))

# Analytic check: fraction touching the tail matches 1-(1-p)^N.
assert abs(touched[100] - (1 - 0.99**100)) < 0.02, "tail amplification formula holds"
# The punchline: at fan-out 100 the MEDIAN request is slower than the
# single-backend p99 — the tail has become the typical experience.
assert medians[100] > p99_single, "median-of-max exceeds single-node p99 at high fan-out"

print(f"single backend p99: {p99_single:.0f} ms")
print(f"{'fan-out':>8} {'median (ms)':>12} {'% slower than single p99':>26}")
for n in fanouts:
    print(f"{n:>8} {medians[n]:>12.0f} {touched[n]:>25.1%}")

fig, ax = plt.subplots()
ax.plot(fanouts, [touched[n] * 100 for n in fanouts], "o-")
ax.set_xscale("log")
ax.set_xlabel("fan-out N (parallel backends per request)")
ax.set_ylabel("% of requests hitting a p99-slow backend")
ax.set_title("Tail latency amplification: 1 − (1 − 0.01)^N")
plt.show()

### 3.3 A classic mitigation: hedged requests

If you cannot make the tail go away, race it. A **hedged request** sends a backup copy of a
call once the original has been slow for a while (say, past the p95), and takes whichever
answer arrives first — trading a few percent extra load for a much shorter tail.

Why does that work? Because real tails are usually not smooth: they come from *transient,
per-server hiccups* — a garbage-collection pause, queueing behind one huge request, a cold
cache. The backup request lands on a different server that is very unlikely to be hiccuping at
the same moment. Let's model latency that way — a fast base plus a 5% chance of a hiccup —
and measure.

In [ ]:
def draw_latency(n: int) -> np.ndarray:
    """Latency = fast lognormal base + occasional transient hiccup.

    The hiccup term is what creates the long tail: 5% of requests get stuck
    behind something (GC, queueing, cold cache) for an exponential extra delay.
    """
    base = rng.lognormal(mean=3.0, sigma=0.4, size=n)
    hiccup = rng.random(n) < 0.05
    return base + hiccup * rng.exponential(scale=150.0, size=n)

n = 200_000
primary = draw_latency(n)
backup = draw_latency(n)  # an independent server: its hiccups are uncorrelated
hedge_at = float(np.percentile(primary, 95))  # fire the backup after this long

# If the primary finishes before the hedge timer, nothing changes. Otherwise the
# effective latency is whichever finishes first: the primary, or the backup
# (which started `hedge_at` ms late).
hedged = np.where(primary <= hedge_at, primary, np.minimum(primary, hedge_at + backup))

p99_before, p99_after = np.percentile(primary, 99), np.percentile(hedged, 99)
extra_load = float(np.mean(primary > hedge_at))

print(f"p99 before: {p99_before:.0f} ms -> after hedging: {p99_after:.0f} ms "
      f"({1 - p99_after / p99_before:.0%} shorter tail) for {extra_load:.0%} extra load")
assert p99_after < 0.6 * p99_before, "hedging should cut the p99 substantially"
assert extra_load < 0.06, "hedging at p95 costs only ~5% duplicate requests"

This pattern — *measure the distribution, then spend a little redundancy exactly where the
distribution is worst* — recurs throughout the course.

### 3.4 Coping with load

Two axes, endless combinations:

- **Scale up** (vertical): a bigger machine. Simple, no distribution problems — until the price
  curve bends and the ceiling arrives.
- **Scale out** (horizontal): more machines. Cheap per unit, but now you own the problems of
  Part 1: replication (nb 05), partitioning (nb 06), and the faults of distribution (nb 08).

Rules of thumb we will keep re-deriving: make services **stateless** (any app server can handle
any request — state lives in the data tier, which is where the hard work concentrates); scale
*for the load parameters you actually have*, because a design optimized for the wrong parameter
scales the wrong thing; and there is no generic "scalable architecture" — only architectures
matched to workloads.

## 4. Maintainability

> **Maintainability**: the ease with which people — future you, new teammates, the 3 a.m.
> on-call — can operate, understand, and change the system.

Most software cost is after launch. Three design targets:

- **Operability** — make life easy for operations: good telemetry (you cannot fix what you
  cannot see), sane defaults, self-healing where safe, no surprises.
- **Simplicity** — fight *accidental complexity* (complexity from the implementation, not the
  problem). The main tool is **abstraction**: clean interfaces hiding messy internals — a
  database hides B-trees; a queue hides retries.
- **Evolvability** — make change safe: schema evolution (nb 04), decoupled producers/consumers
  (nb 11), pipelines you can rebuild from raw data (nb 12).

This course practices what it preaches: deterministic seeds (operability: reproducible
behavior), self-checking asserts (telemetry), small single-purpose builds (simplicity), and a
shared toolkit reused across notebooks (evolvability).

## 5. The design framework used in this course

Every design discussion — and each capstone in Part 3 — follows five steps:

1. **Requirements.** What must be true? Functional (features) and non-functional: target p99,
   availability, consistency needs, cost. Write numbers, not adjectives.
2. **Workload.** Estimate the load parameters: requests/s, read:write ratio, data size, growth,
   fan-out, hot spots. (Notebook 01 builds the estimation toolkit.)
3. **Design.** Compose the standard blocks; state what each guarantees and costs.
4. **Failure analysis.** For every arrow in the diagram: what if it is slow? lost? duplicated?
   stale? What is the blast radius, and is any failure correlated across "redundant" parts?
5. **Evolution.** What growth or requirement change breaks this first, and what is the
   migration path?

A design is not "done" when it works on the happy path; it is done when steps 4 and 5 have
honest answers.

## 6. Exercises

Work these before opening the solutions below — the struggle is the point.

1. **Chains.** Your request path is: load balancer (99.99%) → app tier (99.95%) → database
   (99.9%) → and, for 30% of requests, a payment provider (99.5%). What availability does a
   payment request see? How many hours per year is that down?
2. **Redundancy budget.** You can afford either one 99.99% database server, or three 99.5%
   replicas (assume independence and that one live replica suffices). Which gives better
   availability? What real-world effect could reverse your answer?
3. **Fan-out.** A search page queries 20 index shards in parallel. Each shard has p99 = 120 ms.
   Roughly what fraction of page loads waits ≥ 120 ms on at least one shard? What is that
   fraction if you cut fan-out to 5 by packing more data per shard?
4. **SLOs.** You promise 99.9% availability per 30-day month (an **SLO** — service level
   objective). An incident just burned 25 minutes of downtime. How much **error budget**
   (allowed downtime) remains this month?
5. **Correlated reality.** Give three concrete shared dependencies that could make "five
   independent replicas" fail together, and one mitigation for each.

### Solutions

**1.** Serial composition, payment included: $0.9999 \times 0.9995 \times 0.999 \times 0.995
\approx 0.9934$ → about 58 hours down per year. The chain is only as strong as its weakest
link, and *adding* the payment dependency cost more than everything else combined.

**2.** Three 99.5% replicas: $1 - 0.005^3 \approx 0.9999999$ — vastly better than 99.99% on
paper. Reversal: any correlated failure mode (same rack, same buggy version, same operator)
collapses the three replicas toward a single fault domain, and the single well-engineered
server can win.

**3.** $1 - 0.99^{20} \approx 18\%$ of page loads touch a p99-slow shard. At fan-out 5 it drops
to $1 - 0.99^{5} \approx 4.9\%$. Reducing fan-out is a *latency* strategy, not just a capacity
one.

**4.** Budget: $0.001 \times 30 \times 24 \times 60 = 43.2$ minutes/month. After 25 minutes:
18.2 minutes remain — deploy carefully.

**5.** Examples: same top-of-rack switch (spread across racks/zones); same code version pushed
everywhere at once (staged rollouts); same certificate or config source (canary configs,
diverse renewal times); same cloud region (multi-region); same on-call human at 3 a.m.
(automation + checklists).

In [ ]:
# Numeric spot-checks for the solutions above — loudly verified, like everything here.
ex1 = serial(0.9999, 0.9995, 0.999, 0.995)
assert abs(ex1 - 0.99340) < 0.0005 and 55 < (1 - ex1) * HOURS_PER_YEAR < 61
assert parallel(0.995, 0.995, 0.995) > 0.9999
assert abs((1 - 0.99**20) - 0.182) < 0.001
assert abs((0.001 * 30 * 24 * 60) - 43.2) < 1e-9
print("solution arithmetic verified")

## 7. Takeaways

- **Reliability** = correct service despite faults. Redundancy annihilates *independent* faults
  and is helpless against *correlated* ones (shared dependencies, software bugs, humans) —
  your availability is capped by your least-available shared dependency.
- **Scalability** is a curve, not a badge. Describe load with explicit parameters; describe
  performance with percentiles. Fan-out amplifies the tail by $1-(1-p)^N$; hedging buys the
  tail back for a few percent of load.
- **Maintainability** is where the money goes: operability, simplicity (abstraction),
  evolvability.
- Design with the five-step framework; a design is done when the *failure* and *evolution*
  questions have honest answers.

**Further reading.** DDIA ch. 1; Dean & Barroso, *The Tail at Scale* (CACM 2013); the Google
SRE book, ch. 3–4 (SLOs and error budgets).

**Next:** [01 · Scale & estimation](01-scale-and-estimation.ipynb) — the arithmetic you need
before designing anything: latency numbers, Little's law, and why systems melt down near 100%
utilization.